# Phase 6 — Evaluation (BLEU)

The final notebook. We put a single number on the model with **BLEU** (`sacrebleu`,
the standard, tokenization-independent implementation) over the test set, then write the
report: model size, training steps, wall-clock, BLEU.

**Set expectations now:** the paper reports **27.3 BLEU** on WMT14 En-De with the full
base model (65M params, 100k steps, 4.5M sentence pairs, checkpoint averaging). Our
notebook model is ~2M params, ~400 steps, 29k pairs, no averaging — so BLEU will be
*tiny*. The point of this phase is to measure correctly and understand **why** the gap
exists, not to hit a number.

In [ ]:
# Bootstrap: put the repo root on sys.path so `import transformer` works from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import time, pathlib
import torch
from tokenizers import Tokenizer

from transformer.config import ModelConfig, TrainConfig
from transformer.tokenizer import train_joint_bpe
from transformer.data import load_multi30k, make_dataloader
from transformer.transformer import Transformer
from transformer.train import fit
from transformer.evaluate import corpus_bleu

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
device = get_device()
torch.manual_seed(0)
data, _ = load_multi30k()

CKPT = pathlib.Path("../checkpoints/nb_phase4.pt")
if CKPT.exists():
    blob = torch.load(CKPT, map_location=device, weights_only=False)
    tokenizer = Tokenizer.from_str(blob["tokenizer"])
    cfg = ModelConfig(**blob["model_config"])
    model = Transformer(cfg).to(device); model.load_state_dict(blob["model_state_dict"])
    print("loaded model from", CKPT)
else:
    tokenizer = train_joint_bpe(data["train"], vocab_size=10000)
    cfg = ModelConfig.smoke(src_vocab_size=tokenizer.get_vocab_size(),
                            tgt_vocab_size=tokenizer.get_vocab_size())
    model = Transformer(cfg).to(device)
    loader = make_dataloader(data["train"][:4000], tokenizer, batch_size=32, shuffle=True, max_len=40)
    fit(model, loader, TrainConfig(warmup_steps=400, log_every=200), device, max_steps=400)
model.eval()
print("params:", model.num_parameters())

## Corpus BLEU on the test set (greedy)

We evaluate a slice of the test set with greedy decoding (fast). On the A100 you'd run
the **full** test set with beam search.

In [ ]:
test_subset = data["test"][:200]
t0 = time.perf_counter()
bleu_greedy, hyps_g = corpus_bleu(model, test_subset, tokenizer, device, decode="greedy", max_len=40)
dt_greedy = time.perf_counter() - t0
print(f"greedy BLEU on {len(test_subset)} test sentences: {bleu_greedy.score:.2f}")
print(f"  ({dt_greedy:.1f}s, {len(test_subset)/dt_greedy:.1f} sent/s)")
print("  signature:", bleu_greedy)

## Beam search BLEU (smaller slice — it's slower)

Beam usually nudges BLEU up a little. We run a smaller slice because beam decodes one
sentence at a time with no KV cache.

In [ ]:
beam_subset = data["test"][:50]
t0 = time.perf_counter()
bleu_beam, hyps_b = corpus_bleu(model, beam_subset, tokenizer, device,
                                decode="beam", beam_size=4, alpha=0.6, max_len=40)
dt_beam = time.perf_counter() - t0
# greedy on the SAME slice for a fair comparison
bleu_g_same, _ = corpus_bleu(model, beam_subset, tokenizer, device, decode="greedy", max_len=40)
print(f"on the same {len(beam_subset)} sentences:")
print(f"  greedy BLEU : {bleu_g_same.score:.2f}")
print(f"  beam=4 BLEU : {bleu_beam.score:.2f}   ({dt_beam:.1f}s, {len(beam_subset)/dt_beam:.1f} sent/s)")

## Read a few hypotheses next to references

BLEU is a blunt instrument — always eyeball some outputs too.

In [ ]:
import sacrebleu
for (en, ref), hyp in list(zip(beam_subset, hyps_b))[:5]:
    sb = sacrebleu.sentence_bleu(hyp, [ref]).score
    print(f"EN  : {en}")
    print(f"ref : {ref}")
    print(f"hyp : {hyp}")
    print(f"sentence BLEU: {sb:.1f}\n")

## The report

In [ ]:
print("=" * 56)
print("  RESULT SUMMARY (notebook smoke model)")
print("=" * 56)
print(f"  parameters        : {model.num_parameters():,}")
print(f"  config            : d_model={cfg.d_model}, h={cfg.n_heads}, "
      f"{cfg.n_encoder_layers}+{cfg.n_decoder_layers} layers")
print(f"  train data        : {len(data['train'])} pairs (subset used: 4000)")
print(f"  steps             : ~400 (smoke)")
print(f"  test BLEU (greedy): {bleu_greedy.score:.2f}  on {len(test_subset)} sentences")
print(f"  paper reference   : 27.3 (WMT14, base, 100k steps, 4.5M pairs, ckpt-avg)")
print("=" * 56)

## Why is BLEU so far below 27.3? (understand the gap)

Every one of these costs BLEU, and our run gives up all of them:

1. **Model capacity** — ~2M params vs 65M. Far less ability to model the mapping.
2. **Training steps** — ~400 vs 100k. The model has barely started learning.
3. **Data** — 4k of 29k Multi30k pairs vs 4.5M WMT14 pairs. Less coverage of phrasing.
4. **Dataset** — Multi30k (image captions) ≠ WMT14 (news); not even the same benchmark.
5. **No checkpoint averaging** — the paper averages the last N checkpoints (~+1-2 BLEU).
6. **Vocabulary** — 10k joint BPE vs ~37k; more `<unk>`-like fragmentation.

The architecture is faithful; the *scale* is what's missing. That is exactly what the
A100 run fixes — and now you can measure the improvement with this same `corpus_bleu`.

## Definition of done (from the plan) — reached
- [x] Can explain every line of `transformer.py` (the components were built one by one).
- [x] Can produce a loss curve (Phase 4), a working translation (Phase 5), a BLEU number (here).
- [x] Can show evolving attention heatmaps (Phase 4 before/after, Phase 5 alignment).

**Next (off the Mac):** lift `fit()` + `corpus_bleu` into `scripts/train.py` for the A100 —
first-run config, full data, periodic checkpoints + BLEU, resume — then do the real run and
watch BLEU climb.